# Level 2 — Vision Transformers (ViT-S/16, 선택적으로 Swin-Tiny)

**목표**: ViT (와 선택적으로 Swin-T) 를 직접 구현하고 Multi-task 로 연결하여 Level 1 의 CNN 들과 비교합니다.

**Pretrained 가중치**: ImageNet `.pth` 파일을 본인이 구현한 모델의 `state_dict` 에 로드하는 것은 허용됩니다. 출처를 명시하세요. **`timm` / `torchvision.models` import 는 금지** 입니다.

In [1]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/jjay321-oss/2026-HYU-AUE8088-PA2

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 70 (delta 23), reused 7 (delta 7), pack-reused 33 (from 2)
Receiving objects: 100% (70/70), 82.08 KiB | 4.56 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 

In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.models.vit import vit_small_patch16_224
# from src.models.swin import SwinTiny  # 선택 사항

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
import wandb; wandb.login()   # API key 입력

# wandb 설정 — 비활성화하려면 None
WANDB_PROJECT = "aue8088-pa2"
WANDB_TAGS    = ["level2"]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jjay321 (jjay321-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=d57965e1-c168-4d10-b0e7-4cb39ab7abad
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:01<00:00, 172MB/s]


압축 해제 중...
완료 → ../data/set_a


In [7]:
def load_deit_small_pretrained(model):
    """
    DeiT-S/16 ImageNet-1K pretrained weight를
    직접 구현한 ViT-S/16 모델에 로드한다.

    ImageNet classification head는 task가 다르므로 제외하고,
    Weather / Scene / Time of Day multi-task head는 random init 상태로 둔다.
    """

    url = "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth"

    ckpt = torch.hub.load_state_dict_from_url(
        url,
        map_location="cpu",
        check_hash=True,
    )

    if "model" in ckpt:
        pretrained_state = ckpt["model"]
    else:
        pretrained_state = ckpt

    model_state = model.state_dict()

    new_state = {}
    loaded_keys = []
    skipped_keys = []

    for k, v in pretrained_state.items():
        # ImageNet classification head는 제외
        if k.startswith("head."):
            skipped_keys.append(k)
            continue

        # 내가 구현한 모델과 key 이름, tensor shape이 둘 다 맞는 경우만 로드
        if k in model_state and model_state[k].shape == v.shape:
            new_state[k] = v
            loaded_keys.append(k)
        else:
            skipped_keys.append(k)

    missing, unexpected = model.load_state_dict(new_state, strict=False)

    print("Loaded key count:", len(loaded_keys))
    print("Skipped key count:", len(skipped_keys))
    print("Missing key count:", len(missing))
    print("Unexpected key count:", len(unexpected))

    print("\nFirst 10 loaded keys:")
    for k in loaded_keys[:10]:
        print(k)

    print("\nMissing keys except task head:")
    for k in missing:
        if not k.startswith("head."):
            print(k)

    return {
        "loaded_keys": loaded_keys,
        "skipped_keys": skipped_keys,
        "missing": missing,
        "unexpected": unexpected,
    }

In [8]:
# 선택 사항: 본인 ViT 구현체에 ImageNet pretrained 가중치를 로드하는 절차
#
# 진행 방식:
#   1) 공개된 ViT-S/16 체크포인트 (.pth) 를 다운로드.
#   2) 모델 인스턴스 생성:  model = vit_small_patch16_224()
#   3) 키 매핑 후 로드:
#        pre = torch.load('vit_s16.pth')
#        missing, unexpected = model.load_state_dict(remap(pre), strict=False)
#        # Multi-task head 는 task 종속이므로 random init 유지.
#
# 사용한 체크포인트 출처와 매칭된 키 개수를 리포트에 기재하세요.
#USE_PRETRAINED = False
#model = vit_small_patch16_224().to(device)
USE_PRETRAINED = True
model = vit_small_patch16_224().to(device)

load_info = load_deit_small_pretrained(model)

Downloading: "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth" to /root/.cache/torch/hub/checkpoints/deit_small_patch16_224-cd65a155.pth
100%|██████████| 84.2M/84.2M [00:00<00:00, 107MB/s]


Loaded key count: 102
Skipped key count: 50
Missing key count: 54
Unexpected key count: 0

First 10 loaded keys:
cls_token
pos_embed
patch_embed.proj.weight
patch_embed.proj.bias
blocks.0.norm1.weight
blocks.0.norm1.bias
blocks.0.attn.qkv.weight
blocks.0.attn.qkv.bias
blocks.0.attn.proj.weight
blocks.0.attn.proj.bias

Missing keys except task head:
blocks.0.mlp.0.weight
blocks.0.mlp.0.bias
blocks.0.mlp.3.weight
blocks.0.mlp.3.bias
blocks.1.mlp.0.weight
blocks.1.mlp.0.bias
blocks.1.mlp.3.weight
blocks.1.mlp.3.bias
blocks.2.mlp.0.weight
blocks.2.mlp.0.bias
blocks.2.mlp.3.weight
blocks.2.mlp.3.bias
blocks.3.mlp.0.weight
blocks.3.mlp.0.bias
blocks.3.mlp.3.weight
blocks.3.mlp.3.bias
blocks.4.mlp.0.weight
blocks.4.mlp.0.bias
blocks.4.mlp.3.weight
blocks.4.mlp.3.bias
blocks.5.mlp.0.weight
blocks.5.mlp.0.bias
blocks.5.mlp.3.weight
blocks.5.mlp.3.bias
blocks.6.mlp.0.weight
blocks.6.mlp.0.bias
blocks.6.mlp.3.weight
blocks.6.mlp.3.bias
blocks.7.mlp.0.weight
blocks.7.mlp.0.bias
blocks.7.mlp.3.weig

In [9]:
import torch

model.eval()

x = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    out = model(x)

print(type(out))
print(out.keys())

for k, v in out.items():
    print(k, v.shape)

<class 'dict'>
dict_keys(['weather', 'scene', 'timeofday'])
weather torch.Size([2, 6])
scene torch.Size([2, 3])
timeofday torch.Size([2, 3])


In [ ]:
epochs = 30
optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
losses = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name="level2-vit_s16-pretrained",#f"level2-vit_s16{'-pretrained' if USE_PRETRAINED else ''}",
    config={
        "backbone": "vit_s16", "pretrained": USE_PRETRAINED,
        "pretrained_source": "facebookresearch/deit, deit_small_patch16_224-cd65a155.pth",
        "loaded_key_count": len(load_info["loaded_keys"]),
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16", "pretrained"],#tags=WANDB_TAGS + ["vit_s16"],
)
trainer = MultiTaskTrainer(model, optim, sched, losses, device, TrainConfig(epochs=epochs), logger=logger)

# TODO:
trainer.fit(train_loader, val_loader)
#
# 학습 종료 후:
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", confusion_matrices(val_pred, val_tgt)[a], CLASS_NAMES[a])
logger.finish()
torch.save(model.state_dict(), "../checkpoints/level2_vit_s16.pth")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


[epoch 01/30] train_loss=2.4213  val_avg_MF1=0.4353  per={'weather': 0.2839584982081091, 'scene': 0.38351633030081217, 'timeofday': 0.6384381338742394}


[epoch 02/30] train_loss=2.0207  val_avg_MF1=0.3915  per={'weather': 0.16897696665737905, 'scene': 0.3404582345758816, 'timeofday': 0.6650936033774317}


train e3:  47%|████▋     | 37/79 [00:15<00:15,  2.65it/s, loss=2.3030]

## 분석 (리포트 필수 포함 항목)

1. **CNN vs Transformer**: 동일 epoch 예산 하에서 ResNet-50 (Level 1) 과 ViT-S (Level 2) 의 Avg-MF1 을 비교하세요.
2. **Pretrained vs Scratch**: 약 5천 장 규모의 소규모 데이터셋에서 ImageNet 초기화가 실제로 얼마나 도움이 되는지 정량적으로 보고하세요.
3. **속성별 거동**: ViT 가 ResNet 대비 Weather 와 Time of Day 사이의 오류 분포를 다르게 가져가는지, 그 원인을 가설로 제시하세요.